In [19]:
import pandas as pd
from pathlib import Path
import json

LOADING DATA

In [20]:
# checking if the dat exists
data_path = Path.cwd() / "SO_queries.json"
print(f"Data exist: {data_path.exists()}")
print(f"Check if is a file: {data_path.is_file()}")

Data exist: True
Check if is a file: True


In [21]:
# extracting the SO queries

with open(data_path, 'r') as file:
    data = []
    for line in file:
        data.append(json.loads(line))
    
df = pd.DataFrame(data)
df = df.drop(columns=['question_title', 'answer_body'])
df.head()

,topic,question_body
0,ABSTRACT_CLASSES_AND_INTERFACES,How do I setup a class that represents an inte...
1,ABSTRACT_CLASSES_AND_INTERFACES,Please explain in an easy to understand langua...
2,ABSTRACT_CLASSES_AND_INTERFACES,Can an abstract class have a constructor?\n\nI...
3,ABSTRACT_CLASSES_AND_INTERFACES,The question is in Java why can't I define an ...
4,ABSTRACT_CLASSES_AND_INTERFACES,(1) List&lt;?&gt; myList = new ArrayList&lt;?&...


In [22]:
print(f"Null values check.")
df.isna().sum()

Null values check.


topic            0
question_body    0
dtype: int64

In [23]:
# all topics
topics = df['topic'].unique()
topics

<ArrowStringArray>
[           'ABSTRACT_CLASSES_AND_INTERFACES',
                                 'ALGORITHMS',
                         'ALGORITHM_ANALYSIS',
                                  'ARRAYS_1D',
                                  'ARRAYS_2D',
                                  'AVL_TREES',
                        'BINARY_SEARCH_TREES',
                        'CLASSES_AND_OBJECTS',
                           'CODING_STANDARDS',
                'COMPOSITION_AND_AGGREGATION',
                               'CONTROL_FLOW',
                            'DESIGN_PATTERNS',
                              'DOCUMENTATION',
                             'DYNAMIC_MEMORY',
                              'ENCAPSULATION',
                         'EXCEPTION_HANDLING',
                                    'FILE_IO',
                   'FUNCTIONS_AND_MODULARITY',
                                     'GRAPHS',
                                'HASH_TABLES',
                                'INHERITA

In [24]:
# 3000 was the max amount of data pulled per topic, trying to see if there's undersampled classes

def topics_under_size(df, counts=1000):
    
    topic_counts = df.loc[:,'topic'].value_counts()
    topic_counts_filter = topic_counts < counts
    return topic_counts[topic_counts_filter].sort_values()

topics_under_size(df)

topic
SOLID_PRINCIPLES                74
AVL_TREES                      117
MULTIWAY_TREES                 186
COMPOSITION_AND_AGGREGATION    291
UML_AND_CLASS_DIAGRAMS         430
ENCAPSULATION                  459
Name: count, dtype: int64

In [25]:
# getting the minory class labels
df_dicts = {}
for column in topics_under_size(df).index:
   df_dicts[column] = df[df['topic'] == column]
   
df_dicts

{'SOLID_PRINCIPLES':                   topic                                      question_body
 26053  SOLID_PRINCIPLES  \n  From Wikipedia :\n  \n  The idea was that ...
 26054  SOLID_PRINCIPLES  The open-closed principle states that "Softwar...
 26055  SOLID_PRINCIPLES  As SOLID principles say, it's better to remove...
 26056  SOLID_PRINCIPLES  I know SOLID principles were written for objec...
 26057  SOLID_PRINCIPLES  I'm currently thinking of "How to design an OS...
 ...                 ...                                                ...
 26122  SOLID_PRINCIPLES  I'm trying to test this custom model where if ...
 26123  SOLID_PRINCIPLES  I would like to see/learn how solid principles...
 26124  SOLID_PRINCIPLES  Keeping things tidy and simple, here is my ini...
 26125  SOLID_PRINCIPLES  In the below example Enums do the amount of pr...
 26126  SOLID_PRINCIPLES  I'm working on my own model of repositories an...
 
 [74 rows x 2 columns],
 'AVL_TREES':           topic             

In [26]:
sample = df.sample(n=1)
sample.to_dict('records')[0]

new_df = df.copy()
pd.concat([new_df, pd.DataFrame([sample.to_dict('records')[0]])])

,topic,question_body
0,ABSTRACT_CLASSES_AND_INTERFACES,How do I setup a class that represents an inte...
1,ABSTRACT_CLASSES_AND_INTERFACES,Please explain in an easy to understand langua...
2,ABSTRACT_CLASSES_AND_INTERFACES,Can an abstract class have a constructor?\n\nI...
3,ABSTRACT_CLASSES_AND_INTERFACES,The question is in Java why can't I define an ...
4,ABSTRACT_CLASSES_AND_INTERFACES,(1) List&lt;?&gt; myList = new ArrayList&lt;?&...
...,...,...
33553,VECTORS_AND_DYNAMIC_ARRAYS,I have a class called ChessBoard and one of th...
33554,VECTORS_AND_DYNAMIC_ARRAYS,I was looking at the source code for gcc (out ...
33555,VECTORS_AND_DYNAMIC_ARRAYS,I am debugging some C++ code and I have a real...
33556,VECTORS_AND_DYNAMIC_ARRAYS,I know the Vector class is thread-safe for add...


In [27]:
import ollama

def generate_synth_data(df_examples : pd.DataFrame = None, topic_name= None, batch_gen=50, amount=500, model = 'gemma4:31b-cloud'):
    """
    Generates synthetic data for minority classes.
    """
    count = 0
    with open(Path.cwd() / f'{topic_name}.ndjson', 'a') as j:
        
        while count < amount:
            
            print(f'GENERATING "{topic_name}" samples... progress: {count} / {amount}')
        
            try:
                
                sample = df_examples.sample(n=5).to_dict('records')[0]
                
                response = ollama.chat(model=model,
                            messages=[{'role': 'system', 'content' : f'You are tasked with generating synthetic data in this exact format {{"topic": ..., "question_body" : ...}}. The user will provide examples and from that, you will generate an entirely new question_body under the SAME TOPIC. Everything should be natural sounding. Avoid adding backslashes to keys \\"topics\\" or \\"question_body"\\ is wrong. Use "" around the keys. Generate {batch_gen} copies separated by <SPLT>. Do not add whitespaces or new lines inside the keys.'},
                                    {'role': 'user', 'content' : f'{sample}'}],
                            think=False,
                            options={'temperature' : 0.8})
                
                content = response.message.content
                
                for line in content.split('<SPLT>'):
                    row = json.loads(line.strip())
                    j.write(json.dumps(row) + '\n')
                    j.flush()
                    
                count += batch_gen
            
            except Exception as e:
                print(f'Failed: {e}. Trying again...')

In [28]:
# generating synthetic data to rebalance values
"""
for topic, topic_df in df_dicts.items():
amount = 1000 - len(topic_df)
generate_synth_data(topic_df, topic, batch_gen=50, amount=amount)
"""

'\nfor topic, topic_df in df_dicts.items():\namount = 1000 - len(topic_df)\ngenerate_synth_data(topic_df, topic, batch_gen=50, amount=amount)\n'

In [29]:
df.shape

(33557, 2)

In [36]:
# we can fix some of these with mapping close topics together.
new_df = df.copy()
for topic, topic_df in df_dicts.items():
    
    json_df = pd.read_json(Path.cwd() / f'{topic}.ndjson', lines=1)
    new_df = pd.concat([json_df, new_df])
    
    
new_df.shape

(39064, 2)

In [42]:
new_df['topic'].value_counts()

topic
AVL_TREES                                     1286
COMPOSITION_AND_AGGREGATION                   1268
MULTIWAY_TREES                                1173
UML_AND_CLASS_DIAGRAMS                        1144
SOLID_PRINCIPLES                              1111
ENCAPSULATION                                 1082
ABSTRACT_CLASSES_AND_INTERFACES               1000
ALGORITHMS                                    1000
ALGORITHM_ANALYSIS                            1000
ARRAYS_1D                                     1000
ARRAYS_2D                                     1000
BINARY_SEARCH_TREES                           1000
CLASSES_AND_OBJECTS                           1000
CODING_STANDARDS                              1000
CONTROL_FLOW                                  1000
DESIGN_PATTERNS                               1000
DOCUMENTATION                                 1000
DYNAMIC_MEMORY                                1000
EXCEPTION_HANDLING                            1000
FILE_IO                  

In [44]:
# checking the number of samples per class
topics_under_size(new_df)

Series([], Name: count, dtype: int64)

VISUALIZING DATA (KINDA)

In [46]:

def random_sample_topics(df, number_per_topic=2):
        
    # randomly sample 
    random_samples = df.groupby('topic').sample(n=number_per_topic, random_state=42)

    return random_samples
    
random_sample_topics(new_df)

,topic,question_body
521,ABSTRACT_CLASSES_AND_INTERFACES,I just encountered a behavior I first thought ...
737,ABSTRACT_CLASSES_AND_INTERFACES,I have an interface class similar to:\n\nclass...
1093,ALGORITHMS,I have three points on the circumference of a ...
1692,ALGORITHMS,I'm trying to efficiently list numbers between...
2612,ALGORITHM_ANALYSIS,I was going through some practice problems and...
...,...,...
489,UML_AND_CLASS_DIAGRAMS,I'm struggling to decide between using a visua...
32212,VARIABLES_AND_TYPES,"So I just started learning Java, its literally..."
31958,VARIABLES_AND_TYPES,Hi I'm new to Java and I have following questi...
33411,VECTORS_AND_DYNAMIC_ARRAYS,im trying to get a vector of integers by using...


NOTE: Due to the nature of our data (heavily code-based), common cleaning techniques can serve more to hinder than help. According to the average sample, there are a lot of new lines. These are likely the things that need the most attention.

In [52]:
import html

cleaned_df = new_df.copy()

# decode html code back into regular characters
cleaned_df = cleaned_df.apply(html.unescape)

# if the string pattern matches newline or carriage return (\r) ONE or MORE time, replace with space
cleaned_df = cleaned_df.replace(r'[\n\r]+', ' ', regex=True)

In [53]:
random_sample_topics(cleaned_df)

,topic,question_body
521,ABSTRACT_CLASSES_AND_INTERFACES,I just encountered a behavior I first thought ...
737,ABSTRACT_CLASSES_AND_INTERFACES,I have an interface class similar to: class II...
1093,ALGORITHMS,I have three points on the circumference of a ...
1692,ALGORITHMS,I'm trying to efficiently list numbers between...
2612,ALGORITHM_ANALYSIS,I was going through some practice problems and...
...,...,...
489,UML_AND_CLASS_DIAGRAMS,I'm struggling to decide between using a visua...
32212,VARIABLES_AND_TYPES,"So I just started learning Java, its literally..."
31958,VARIABLES_AND_TYPES,Hi I'm new to Java and I have following questi...
33411,VECTORS_AND_DYNAMIC_ARRAYS,im trying to get a vector of integers by using...


In [55]:
# cleaned df relable

cleaned_df.columns = {'label' : 'topic', 'content' : 'question_body'}
cleaned_df

,label,content
0,ENCAPSULATION,I am trying to implement a plugin system where...
1,ENCAPSULATION,Suppose I have a class hierarchy where 'Shape'...
2,ENCAPSULATION,I'm working with a library that uses a base cl...
3,ENCAPSULATION,"In C++, if I have a base class pointer pointin..."
4,ENCAPSULATION,I have a base class 'Entity' and derived class...
...,...,...
33552,VECTORS_AND_DYNAMIC_ARRAYS,Looks like a stupid question. But comment to m...
33553,VECTORS_AND_DYNAMIC_ARRAYS,I have a class called ChessBoard and one of th...
33554,VECTORS_AND_DYNAMIC_ARRAYS,I was looking at the source code for gcc (out ...
33555,VECTORS_AND_DYNAMIC_ARRAYS,I am debugging some C++ code and I have a real...


In [56]:
# saving to parquet
save_path = Path.cwd() / "train_ready.parquet"
cleaned_df.to_parquet(path=save_path)

if save_path.is_file():
    
    try:
        test = pd.read_parquet(save_path)
        
        print('If all zero, then file integrity is good.')
        print((test != cleaned_df).sum())
        
        print(f'Successfully saved: {save_path}')
        
    except Exception as e:
        print('Save Failed.', {e})
else:
    print('Save failed.')

If all zero, then file integrity is good.
label      0
content    0
dtype: int64
Successfully saved: c:\All University Materials\Project\ICT304-project\ai\ai_tools\data_preprocessing\train_ready.parquet
